# Controlled changes and FINAL configurations

The five single-factor conditions change segmentation, embedding model or entity
types relative to the corresponding baseline. FINAL combines selected factors.
Each condition retains its system's native retrieval and answer path.

Read the two baseline system paths first. This notebook runs without credentials
or model requests. It identifies effective settings, index reuse and the stored
artefacts needed to inspect a condition. The same configurable pipeline notebooks
execute those conditions. See thesis Section 2.8, Table 6, and Section 3.4.

In [1]:
import hashlib
import json
from pathlib import Path

from IPython.display import Markdown, display
from tabulate import tabulate

from rag_comparison import config
from rag_comparison.artifacts import verify_artifacts

## Scientific conditions

`small` and `large` refer to the two Azure embedding models at the same 1536
dimensions. Page segmentation uses each complete prepared page. Structure
segmentation uses source-exact page intervals, at most 800 `o200k_base` tokens
and zero overlap. ENTITY extends the four baseline types to nine types.

In [2]:
rows = []
for condition_id, state in config.CONDITIONS.items():
    segmentation = (
        "page"
        if state["chunking_policy"] == config.PAGE_CHUNKING_POLICY
        else "structure / 800"
    )
    embedding = state["embedding_model"].removeprefix("text-embedding-3-")
    entities = len(state["entity_types"]) if state["entity_types"] else None
    graph_source = state["graph_source"] or (
        "Standard Indexing" if state["system"] == "G" else "vector index"
    )
    rows.append(
        (
            condition_id,
            segmentation,
            embedding,
            config.EMBEDDING_DIMENSION,
            entities,
            graph_source,
        )
    )
display(
    Markdown(
        tabulate(
            rows,
            headers=[
                "Condition",
                "Segmentation",
                "Embedding",
                "Dimensions",
                "Entity types",
                "Index source / method",
            ],
            tablefmt="github",
        )
    )
)

| Condition                    | Segmentation    | Embedding   |   Dimensions |   Entity types | Index source / method        |
|------------------------------|-----------------|-------------|--------------|----------------|------------------------------|
| C0                           | page            | small       |         1536 |                | vector index                 |
| G0                           | page            | small       |         1536 |              4 | Standard Indexing            |
| C-CHUNK-STRUCT-800-0         | structure / 800 | small       |         1536 |                | vector index                 |
| G-CHUNK-STRUCT-800-0         | structure / 800 | small       |         1536 |              4 | Standard Indexing            |
| C-EMBED-LARGE-1536           | page            | large       |         1536 |                | vector index                 |
| G-EMBED-LARGE-1536           | page            | large       |         1536 |              4 | G0                           |
| G-ENTITY-POLICY-PROCESS-ROLE | page            | small       |         1536 |              9 | Standard Indexing            |
| C-FINAL                      | structure / 800 | large       |         1536 |                | vector index                 |
| G-FINAL                      | page            | large       |         1536 |              9 | G-ENTITY-POLICY-PROCESS-ROLE |

## Check each change against its baseline

The comparison uses the three scientific factors. Operative throughput and retries
are shown separately. Each single-factor condition differs in exactly one factor;
FINAL differs in two. Its selected combination is outcome-informed, with no
independent holdout or interaction estimate.

In [3]:
factors = ("chunking_policy", "embedding_model", "entity_types")
differences = []
for condition_id, state in config.CONDITIONS.items():
    if condition_id in ("C0", "G0"):
        continue
    reference_id = state["system"] + "0"
    baseline = config.CONDITIONS[reference_id]
    changed = [factor for factor in factors if state[factor] != baseline[factor]]
    assert len(changed) == (2 if condition_id.endswith("FINAL") else 1)
    differences.append((condition_id, reference_id, changed))
assert config.CONDITIONS["G-FINAL"]["chunking_policy"] == config.PAGE_CHUNKING_POLICY
display(
    Markdown(
        tabulate(
            differences,
            headers=["Condition", "Reference", "Changed scientific factors"],
            tablefmt="github",
        )
    )
)

| Condition                    | Reference   | Changed scientific factors             |
|------------------------------|-------------|----------------------------------------|
| C-CHUNK-STRUCT-800-0         | C0          | ['chunking_policy']                    |
| G-CHUNK-STRUCT-800-0         | G0          | ['chunking_policy']                    |
| C-EMBED-LARGE-1536           | C0          | ['embedding_model']                    |
| G-EMBED-LARGE-1536           | G0          | ['embedding_model']                    |
| G-ENTITY-POLICY-PROCESS-ROLE | G0          | ['entity_types']                       |
| C-FINAL                      | C0          | ['chunking_policy', 'embedding_model'] |
| G-FINAL                      | G0          | ['embedding_model', 'entity_types']    |

## CHUNK uses a shared source-exact population

[Structure segmentation](04a_structure_chunking.ipynb) constructs the same segment
population for C-CHUNK, G-CHUNK and C-FINAL. It detects document-confirmed headings
and special structures, builds a Docling representation, uses the unchanged native
`HybridChunker`, and binds the output back to exact source intervals.
Each segment retains its original page Evidence Unit. Several retrieved segments
can therefore share a page; duplicate page positions are not backfilled in nDCG.

In [4]:
structure_dir = Path("../examples/structure")
structure_manifest = json.loads(
    (structure_dir / "run_manifest.json").read_text(encoding="utf-8")
)
structure_paths = verify_artifacts(structure_dir, structure_manifest)
segments = [
    json.loads(line)
    for line in structure_paths["segments"].read_text(encoding="utf-8").splitlines()
    if line
]
print(
    {
        "source_pages": structure_manifest["source_page_count"],
        "segments": len(segments),
        "maximum_tokens": max(row["token_count"] for row in segments),
        "overlap": config.STRUCTURE_POLICY["OVERLAP"],
    }
)
display(
    Markdown(
        tabulate(
            [
                (
                    row["page_number"],
                    row["character_start"],
                    row["character_end"],
                    row["token_count"],
                )
                for row in segments[:5]
            ],
            headers=["Page", "Character start", "Character end", "Tokens"],
            tablefmt="github",
        )
    )
)

{'source_pages': 20, 'segments': 44, 'maximum_tokens': 687, 'overlap': 0}


|   Page |   Character start |   Character end |   Tokens |
|--------|-------------------|-----------------|----------|
|      1 |                 0 |            2103 |      687 |
|      2 |                 0 |             447 |      182 |
|      2 |               447 |            1946 |      298 |
|      2 |              1946 |            2520 |      116 |
|      3 |                 0 |            2150 |      449 |

## EMBED replaces vectors while holding other factors fixed

Classical RAG recomputes both document and query vectors using `large`, retains
1536 dimensions and uses the same exact stable cosine retrieval. GraphRAG preserves
the six native graph tables and recomputes its three vector collections with
`generate_text_embeddings`. G-EMBED uses G0 tables; G-FINAL uses G-ENTITY tables.
A distinct target vector store protects the source index from native overwrite.

In [5]:
display(
    Markdown(
        tabulate(
            [
                (model, variable, config.EMBEDDING_DIMENSION)
                for model, variable in config.EMBEDDING_DEPLOYMENT_VARIABLES.items()
            ],
            headers=["Embedding model", "Azure deployment variable", "Dimensions"],
            tablefmt="github",
        )
    )
)
print(
    {
        "native_embedding_workflow": config.GRAPHRAG_EMBED_ONLY_WORKFLOWS,
        "vector_collections": config.GRAPHRAG_VECTOR_COLLECTIONS,
    }
)

| Embedding model        | Azure deployment variable               |   Dimensions |
|------------------------|-----------------------------------------|--------------|
| text-embedding-3-small | AZURE_OPENAI_EMBEDDING_DEPLOYMENT       |         1536 |
| text-embedding-3-large | AZURE_OPENAI_EMBEDDING_LARGE_DEPLOYMENT |         1536 |

{'native_embedding_workflow': ['generate_text_embeddings'], 'vector_collections': ['text_unit_text', 'entity_description', 'community_full_content']}


## ENTITY extends the extraction type list

The native extraction prompt receives the nine-type list below. No extra type
definitions, examples or few-shot instructions are added. Entities, relationships,
communities and reports must be rebuilt. The other scientific factors retain G0
settings; FINAL keeps these G-ENTITY graph records and replaces their embeddings.

In [6]:
display(
    Markdown(
        tabulate(
            [
                (name, "baseline" if name in config.GRAPHRAG_ENTITY_TYPES else "added")
                for name in config.EXTENDED_ENTITY_TYPES
            ],
            headers=["Entity type", "Membership"],
            tablefmt="github",
        )
    )
)
print(
    {
        "baseline_concurrency": config.GRAPHRAG_CONCURRENT_REQUESTS,
        "ENTITY_index_concurrency": config.GRAPHRAG_ENTITY_CONCURRENT_REQUESTS,
        "variant_native_max_retries": config.GRAPHRAG_VARIANT_MAX_RETRIES,
    }
)

| Entity type          | Membership   |
|----------------------|--------------|
| organization         | baseline     |
| person               | baseline     |
| geo                  | baseline     |
| event                | baseline     |
| policy               | added        |
| process              | added        |
| role                 | added        |
| technical_system     | added        |
| information_artifact | added        |

{'baseline_concurrency': 5, 'ENTITY_index_concurrency': 25, 'variant_native_max_retries': 7}


## Execute through the existing system path

Set `CONDITION_ID` in [Classical RAG](02_classical_rag.ipynb) or
[GraphRAG indexing](03_graphrag_index.ipynb). Select the condition's prepared
input folder and a distinct ignored output folder. CHUNK and C-FINAL use the
structure notebook's output; other conditions use page inputs. For G-EMBED or
G-FINAL also select the completed `source_index_dir` required above.

Keep execution controls false for stored-result inspection. Explicit Azure
generation selects the same model and dimension in the request, checkpoint,
configuration and index sidecar. [Local Search](03_graphrag_local_search.ipynb)
reads its condition from the selected index manifest. Native response contracts
are retained for every condition. See [the execution and artefact notes](../docs/controlled_changes.md).

## Inspect a selected stored index

The public G0 index is selected below. An approved private variant uses the same
relative file bindings in an ignored executed notebook copy. EMBED-only manifests
retain the source index identity. Selecting its source folder also checks all six
Parquet tables for byte equality. This step reads files; it does not rebuild an index.

In [7]:
stored_dir = Path("../examples/graphrag")
stored_source_dir = None
stored_manifest = json.loads(
    (stored_dir / "index_manifest.json").read_text(encoding="utf-8")
)
stored_paths = verify_artifacts(stored_dir, stored_manifest)
stored_state = config.CONDITIONS[stored_manifest["condition_id"]]
configuration = json.loads(stored_paths["configuration"].read_text(encoding="utf-8"))
assert (
    configuration["settings"]["embedding_models"]["default_embedding_model"]["model"]
    == stored_state["embedding_model"]
)
assert (
    configuration["settings"]["extract_graph"]["entity_types"]
    == stored_state["entity_types"]
)
print(
    {
        "stored_condition": stored_manifest["condition_id"],
        "embedding_model": stored_state["embedding_model"],
        "tables": 6,
        "version": stored_manifest["graphrag_version"],
    }
)
if stored_state["graph_source"] is not None:
    assert stored_source_dir is not None, "Select the completed source index folder."
    source_manifest = json.loads(
        (stored_source_dir / "index_manifest.json").read_text(encoding="utf-8")
    )
    source_paths = verify_artifacts(stored_source_dir, source_manifest)
    assert source_manifest["condition_id"] == stored_state["graph_source"]
    assert (
        hashlib.sha256(
            (stored_source_dir / "index_manifest.json").read_bytes()
        ).hexdigest()
        == stored_manifest["graph_source"]["index_manifest_sha256"]
    )
    for name in (
        "documents",
        "text_units",
        "entities",
        "relationships",
        "communities",
        "community_reports",
    ):
        assert stored_paths[name].read_bytes() == source_paths[name].read_bytes()

{'stored_condition': 'G0', 'embedding_model': 'text-embedding-3-small', 'tables': 6, 'version': '3.1.0'}


## Relate a condition to its reported evaluation

Section 3.4.1 reports the five single-factor changes against their respective
baselines. Section 3.4.2 reports FINAL combinations on their own expanded common
relevance pool. PRIMARY and FINAL nDCG values therefore need their matching qrels
and IDCGs; they are not interchangeable. Newly generated answers require new
accepted judgements. Stored responses and accepted judgements remain separate
inputs in the evaluation notebooks. The public source annotations describe the
RFC example and are not private thesis judgements.